In [ ]:
!pip install -q pytorch-crf tqdm pandas==2.2.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 104.6 MB/s eta 0:00:0000:01:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


In [ ]:
import os
import random
import re
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from torchcrf import CRF
from tqdm.auto import tqdm


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# SEEDS = [42, 1337, 3407, 1234, 2026]
SEEDS = [42]

TRAINING_PARAMS = {
    "batch_size": 128,
    "epochs": 5,
    "lr": 2e-3,
}

MODEL_PARAMS = {
    "emb_dim": 160,
    "type_dim": 16,
    "hid_dim": 256,
    "dropout": 0.25,
    "lstm_layers": 2,
}

CONSTRAINT_PARAMS = {
    "max_len": 256,
    "stride": 192,
    "grad_clip_norm": 5.0,
    "num_char_types": 6,
}

PATHS = {
    "test": "/kaggle/input/competitions/super-ai-engineer-ss-6-word-segmentation/ws_test.txt",
    "sample_submission": "/kaggle/input/competitions/super-ai-engineer-ss-6-word-segmentation/ws_sample_submission.csv",
    "output": "submission_bilstm_crf_seed.csv",
}

TAG2ID = {"B_WORD": 0, "I_WORD": 1, "E_WORD": 2}
ID2TAG = {v: k for k, v in TAG2ID.items()}


In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def find_lst20_base() -> Path:
    for p in [
        "/kaggle/input/datasets/nonbangkok/lst20-corpus-nonbangkok/LST20_Corpus",
    ]:
        if os.path.exists(p):
            return Path(p)
    raise FileNotFoundError("LST20_Corpus not found")


def word_to_labels(word: str):
    n = len(word)
    if n == 1:
        return ["B_WORD"]
    if n == 2:
        return ["B_WORD", "E_WORD"]
    return ["B_WORD"] + ["I_WORD"] * (n - 2) + ["E_WORD"]


def char_type(ch: str):
    if "฀" <= ch <= "๿":
        return 1
    if ch.isdigit():
        return 2
    if "a" <= ch.lower() <= "z":
        return 3
    if ch in '.,;:!?()[]{}"\'/-_+=*&^%$#@~`|\\<>':
        return 4
    return 5


def read_split(split_dir: Path):
    data = []
    files = [
        p
        for p in sorted(split_dir.rglob("*"))
        if p.is_file() and not p.name.startswith("._") and p.suffix.lower() in {".txt", ".tsv"}
    ]

    for fp in tqdm(files, desc=f"read {split_dir.name}"):
        chars, tags = [], []
        with open(fp, "r", encoding="utf-8-sig") as f:
            for line in f:
                line = line.rstrip("\n")
                if not line.strip():
                    if chars:
                        data.append((chars, tags))
                        chars, tags = [], []
                    continue

                tok = line.split("\t")[0].strip()
                if not tok:
                    continue

                if tok == "_":
                    if chars:
                        data.append((chars, tags))
                        chars, tags = [], []
                else:
                    chars.extend(list(tok))
                    tags.extend(word_to_labels(tok))

        if chars:
            data.append((chars, tags))

    return data


def chunk_data(samples):
    xs, ys = [], []
    max_len = CONSTRAINT_PARAMS["max_len"]
    for chars, tags in tqdm(samples, desc="chunk"):
        for i in range(0, len(chars), max_len):
            xs.append(chars[i : i + max_len])
            ys.append(tags[i : i + max_len])
    return xs, ys


In [ ]:
BASE = find_lst20_base()
samples = read_split(BASE / "train") + read_split(BASE / "eval")
train_x, train_y = chunk_data(samples)

char2id = {"<pad>": 0, "<unk>": 1}
for ch in Counter(c for seq in train_x for c in seq):
    char2id[ch] = len(char2id)


class SegDS(Dataset):
    def __init__(self, xs, ys):
        self.xs = [torch.tensor([char2id.get(c, 1) for c in x], dtype=torch.long) for x in xs]
        self.ts = [torch.tensor([char_type(c) for c in x], dtype=torch.long) for x in xs]
        self.ys = [torch.tensor([TAG2ID[t] for t in y], dtype=torch.long) for y in ys]

    def __len__(self):
        return len(self.xs)

    def __getitem__(self, i):
        return self.xs[i], self.ts[i], self.ys[i]


def collate(batch):
    xs, ts, ys = zip(*batch)
    lens = torch.tensor([len(x) for x in xs], dtype=torch.long)
    xs = pad_sequence(xs, batch_first=True, padding_value=0)
    ts = pad_sequence(ts, batch_first=True, padding_value=0)
    ys = pad_sequence(ys, batch_first=True, padding_value=0)
    mask = torch.arange(xs.size(1)).unsqueeze(0) < lens.unsqueeze(1)
    return xs.to(DEVICE), ts.to(DEVICE), ys.to(DEVICE), mask.to(DEVICE)


loader = DataLoader(
    SegDS(train_x, train_y),
    batch_size=TRAINING_PARAMS["batch_size"],
    shuffle=True,
    collate_fn=collate,
)


read train:   0%|          | 0/3794 [00:00<?, ?it/s]

read eval:   0%|          | 0/474 [00:00<?, ?it/s]

chunk:   0%|          | 0/511230 [00:00<?, ?it/s]

In [ ]:
class BiLSTMCRF(nn.Module):
    def __init__(self, vocab_size, num_tags):
        super().__init__()
        self.char_emb = nn.Embedding(vocab_size, MODEL_PARAMS["emb_dim"], padding_idx=0)
        self.type_emb = nn.Embedding(CONSTRAINT_PARAMS["num_char_types"], MODEL_PARAMS["type_dim"], padding_idx=0)
        self.lstm = nn.LSTM(
            MODEL_PARAMS["emb_dim"] + MODEL_PARAMS["type_dim"],
            MODEL_PARAMS["hid_dim"] // 2,
            num_layers=MODEL_PARAMS["lstm_layers"],
            bidirectional=True,
            batch_first=True,
            dropout=MODEL_PARAMS["dropout"],
        )
        self.drop = nn.Dropout(MODEL_PARAMS["dropout"])
        self.fc = nn.Linear(MODEL_PARAMS["hid_dim"], num_tags)
        self.crf = CRF(num_tags, batch_first=True)

    def emissions(self, x, t):
        z = torch.cat([self.char_emb(x), self.type_emb(t)], dim=-1)
        z, _ = self.lstm(z)
        return self.fc(self.drop(z))

    def forward(self, x, t, y, mask):
        return -self.crf(self.emissions(x, t), y, mask=mask, reduction="mean")

    def decode(self, x, t, mask):
        return self.crf.decode(self.emissions(x, t), mask=mask)


def decode_once(model, chars):
    x = torch.tensor([[char2id.get(c, 1) for c in chars]], dtype=torch.long, device=DEVICE)
    t = torch.tensor([[char_type(c) for c in chars]], dtype=torch.long, device=DEVICE)
    mask = torch.ones_like(x, dtype=torch.bool, device=DEVICE)
    return model.decode(x, t, mask)[0]


def decode_overlap(model, chars):
    n = len(chars)
    max_len = CONSTRAINT_PARAMS["max_len"]
    stride = CONSTRAINT_PARAMS["stride"]

    if n <= max_len:
        return decode_once(model, chars)

    starts = list(range(0, n - max_len + 1, stride))
    if starts[-1] != n - max_len:
        starts.append(n - max_len)

    votes = [[0, 0, 0] for _ in range(n)]
    for s in starts:
        pred = decode_once(model, chars[s : s + max_len])
        for i, p in enumerate(pred):
            votes[s + i][p] += 1

    return [max(range(3), key=lambda k: v[k]) for v in votes]


In [ ]:
with open(PATHS["test"], "r", encoding="utf-8") as f:
    test_text = f.read()

spans = [(m.start(), m.end(), list(m.group())) for m in re.finditer(r"\S+", test_text)]
ensemble_votes = [[0, 0, 0] for _ in range(len(test_text))]

for seed in SEEDS:
    print(f"\n=== seed {seed} ===")
    set_seed(seed)

    model = BiLSTMCRF(len(char2id), len(TAG2ID)).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=TRAINING_PARAMS["lr"])

    for epoch in range(TRAINING_PARAMS["epochs"]):
        model.train()
        pbar = tqdm(loader, desc=f"seed {seed} epoch {epoch + 1}/{TRAINING_PARAMS['epochs']}")
        for x, t, y, mask in pbar:
            optimizer.zero_grad()
            loss = model(x, t, y, mask)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONSTRAINT_PARAMS["grad_clip_norm"])
            optimizer.step()
            pbar.set_postfix(loss=f"{loss.item():.4f}")

    model.eval()
    with torch.no_grad():
        for s, e, chars in tqdm(spans, desc=f"predict seed {seed}"):
            pred = decode_overlap(model, chars)
            for pos, p in zip(range(s, e), pred):
                ensemble_votes[pos][p] += 1



=== seed 42 ===


seed 42 epoch 1/5:   0%|          | 0/3996 [00:00<?, ?it/s]

In [ ]:
labels_full = ["B_WORD"] * len(test_text)
for i, v in enumerate(ensemble_votes):
    if sum(v):
        labels_full[i] = ID2TAG[max(range(3), key=lambda k: v[k])]

sub = pd.read_csv(PATHS["sample_submission"])
offset = 0 if sub["Id"].min() == 0 else 1
sub["Predicted"] = [labels_full[i - offset] for i in sub["Id"]]
sub.to_csv(PATHS["output"], index=False)

print(sub.head())
print("saved ->", PATHS["output"])
